# Global Power Plant Database — Analysis
### NumPy, Pandas, and Matplotlib Integration

In [ ]:
import io, zipfile, requests
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

sns.set_theme(style='whitegrid', palette='muted')

URL = (
    "https://github.com/devtlv/Datasets-DA-Bootcamp-2-/raw/refs/heads/main/"
    "Week%206%20-%20Applications%20for%20Data%20Analysis/W6D2%20-%20Advanced%20Numpy/"
    "globalpowerplantdatabasev130.zip"
)

try:
    r = requests.get(URL, timeout=30)
    with zipfile.ZipFile(io.BytesIO(r.content)) as z:
        csv_name = [f for f in z.namelist() if f.endswith('.csv')][0]
        with z.open(csv_name) as f:
            df = pd.read_csv(f)
    print("Loaded from URL.")
except Exception as e:
    print(f"URL failed ({e}). Falling back to local file.")
    df = pd.read_csv('global_power_plant_database.csv')

print(f"Shape: {df.shape}")
df.head()

## 1. Data Import and Cleaning

In [ ]:
print("=== Dataset Info ===")
df.info()
print("\n=== Missing values (count and %) ===")
missing = pd.DataFrame({
    'count': df.isnull().sum(),
    'pct':   (df.isnull().mean() * 100).round(2)
})
print(missing[missing['count'] > 0].sort_values('pct', ascending=False))

In [ ]:
# Keep only the columns we need
cols = [
    'country', 'country_long', 'name', 'primary_fuel',
    'capacity_mw', 'latitude', 'longitude', 'commissioning_year'
]
df = df[cols].copy()

# Convert to numeric with NumPy
for col in ['capacity_mw', 'latitude', 'longitude', 'commissioning_year']:
    df[col] = pd.to_numeric(df[col], errors='coerce')

# Drop rows with missing capacity (core metric)
df = df.dropna(subset=['capacity_mw'])

# Fill commissioning_year with median
df['commissioning_year'] = df['commissioning_year'].fillna(
    df['commissioning_year'].median()
).astype(int)

print(f"Clean dataset shape: {df.shape}")
print(f"Remaining missing values:\n{df.isnull().sum()}")

## 2. Exploratory Data Analysis

In [ ]:
print("=== Key Statistics ===")
print(df[['capacity_mw', 'commissioning_year']].describe().round(2))

In [ ]:
top_countries = df['country_long'].value_counts().head(15)
fuel_counts   = df['primary_fuel'].value_counts()

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

top_countries[::-1].plot(kind='barh', ax=axes[0], color='steelblue', edgecolor='white')
axes[0].set_title('Top 15 Countries by Number of Power Plants')
axes[0].set_xlabel('Count')
axes[0].grid(axis='x', alpha=0.3)

fuel_counts.plot(kind='bar', ax=axes[1], color='coral', edgecolor='white')
axes[1].set_title('Distribution by Primary Fuel Type')
axes[1].set_xlabel('Fuel Type')
axes[1].set_ylabel('Count')
axes[1].tick_params(axis='x', rotation=45)
axes[1].grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.show()

## 3. Statistical Analysis

In [ ]:
top_fuels = df['primary_fuel'].value_counts().head(8).index.tolist()
fuel_stats = (
    df[df['primary_fuel'].isin(top_fuels)]
    .groupby('primary_fuel')['capacity_mw']
    .agg(['mean', 'median', 'std', 'count'])
    .round(2)
    .sort_values('mean', ascending=False)
)
print("Capacity (MW) statistics by fuel type:")
print(fuel_stats)

In [ ]:
# Hypothesis test: do Solar and Wind differ in mean capacity?
# Using a two-sample t-test computed manually with NumPy

solar = df[df['primary_fuel'] == 'Solar']['capacity_mw'].dropna().values
wind  = df[df['primary_fuel'] == 'Wind']['capacity_mw'].dropna().values

mean_diff = np.mean(solar) - np.mean(wind)
se = np.sqrt(np.var(solar, ddof=1) / len(solar) + np.var(wind, ddof=1) / len(wind))
t_stat = mean_diff / se

print(f"Solar — mean capacity: {np.mean(solar):.2f} MW  (n={len(solar)})")
print(f"Wind  — mean capacity: {np.mean(wind):.2f} MW  (n={len(wind)})")
print(f"t-statistic: {t_stat:.4f}")
if abs(t_stat) > 1.96:
    print("Conclusion: The mean capacities differ significantly (reject H0 at 95% confidence).")
else:
    print("Conclusion: No significant difference detected (fail to reject H0).")

## 4. Time Series Analysis

In [ ]:
yearly = (
    df[df['commissioning_year'].between(1950, 2022)]
    .groupby('commissioning_year')['capacity_mw']
    .agg(['sum', 'count'])
    .reset_index()
)

fig, axes = plt.subplots(2, 1, figsize=(14, 8), sharex=True)

axes[0].plot(yearly['commissioning_year'], yearly['sum'] / 1e3,
             color='steelblue', linewidth=1.5)
axes[0].fill_between(yearly['commissioning_year'], yearly['sum'] / 1e3, alpha=0.15, color='steelblue')
axes[0].set_title('Total Capacity Added per Year (GW)')
axes[0].set_ylabel('Capacity (GW)')
axes[0].grid(True, alpha=0.3)

axes[1].bar(yearly['commissioning_year'], yearly['count'],
            color='coral', edgecolor='none', width=0.8)
axes[1].set_title('Number of Plants Commissioned per Year')
axes[1].set_xlabel('Year')
axes[1].set_ylabel('Count')
axes[1].grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# Fuel mix evolution over decades
df['decade'] = (df['commissioning_year'] // 10 * 10).astype(int)

fuel_decade = (
    df[df['primary_fuel'].isin(top_fuels)]
    .groupby(['decade', 'primary_fuel'])['capacity_mw']
    .sum()
    .unstack(fill_value=0)
)
fuel_decade = fuel_decade[fuel_decade.index.between(1950, 2020)]

fuel_decade.plot(kind='bar', stacked=True, figsize=(14, 5),
                 colormap='tab10', edgecolor='none')
plt.title('Capacity Added by Fuel Type per Decade (MW)')
plt.xlabel('Decade')
plt.ylabel('Capacity (MW)')
plt.legend(title='Fuel', bbox_to_anchor=(1.01, 1), loc='upper left', fontsize=8)
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

## 5. Advanced Visualization

In [ ]:
# Box plot: capacity distribution by fuel type
plot_df = df[df['primary_fuel'].isin(top_fuels)].copy()
plot_df['log_capacity'] = np.log1p(plot_df['capacity_mw'])

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

sns.boxplot(data=plot_df, x='primary_fuel', y='log_capacity',
            ax=axes[0], palette='muted')
axes[0].set_title('Capacity Distribution by Fuel Type (log scale)')
axes[0].set_xlabel('Fuel Type')
axes[0].set_ylabel('log(1 + Capacity MW)')
axes[0].tick_params(axis='x', rotation=45)

# Geographical scatter
geo = df.dropna(subset=['latitude', 'longitude'])
fuel_colors = {
    'Solar': 'gold', 'Wind': 'skyblue', 'Hydro': 'royalblue',
    'Gas': 'orange', 'Coal': 'dimgray', 'Nuclear': 'limegreen',
    'Oil': 'saddlebrown', 'Biomass': 'olivedrab'
}
for fuel in top_fuels:
    subset = geo[geo['primary_fuel'] == fuel]
    axes[1].scatter(
        subset['longitude'], subset['latitude'],
        s=1, alpha=0.4, label=fuel,
        color=fuel_colors.get(fuel, 'grey')
    )
axes[1].set_title('Geographical Distribution of Power Plants')
axes[1].set_xlabel('Longitude')
axes[1].set_ylabel('Latitude')
axes[1].legend(markerscale=5, fontsize=8, loc='lower left')
axes[1].grid(True, alpha=0.2)

plt.tight_layout()
plt.show()

## 6. Matrix Operations in Real-World Context

In [ ]:
# Build a feature matrix: capacity, latitude, longitude (standardized)
features = df[['capacity_mw', 'latitude', 'longitude']].dropna()
X = features.values.astype(float)

# Standardize with NumPy
X_std = (X - X.mean(axis=0)) / X.std(axis=0)

# Covariance matrix
cov_matrix = np.cov(X_std.T)
print("Covariance matrix (standardized features):")
print(cov_matrix.round(4))

# Eigenvalues and eigenvectors
eigenvalues, eigenvectors = np.linalg.eig(cov_matrix)
print("\nEigenvalues:")
print(eigenvalues.round(4))
print("\nEigenvectors (columns = principal directions):")
print(eigenvectors.round(4))

explained = eigenvalues / eigenvalues.sum() * 100
print("\nVariance explained per component (%):", explained.round(2))
print("""
Interpretation:
  Eigenvalues indicate how much variance each principal component captures.
  A large eigenvalue means the corresponding eigenvector direction carries
  most of the information in the dataset. This is the foundation of PCA.
""")

## 7. Integrating NumPy with Pandas and Matplotlib

In [ ]:
# NumPy-powered filtering in Pandas: keep plants with capacity > mean + 2*std
cap = df['capacity_mw'].values
threshold = np.mean(cap) + 2 * np.std(cap)
large_plants = df[np.array(df['capacity_mw'] > threshold)].copy()

print(f"Threshold (mean + 2*std): {threshold:.1f} MW")
print(f"Large power plants: {len(large_plants)} ({len(large_plants)/len(df)*100:.1f}% of total)")
print(large_plants[['name', 'country_long', 'primary_fuel', 'capacity_mw']]
      .sort_values('capacity_mw', ascending=False).head(10).to_string(index=False))

In [ ]:
# NumPy-enhanced Matplotlib: histogram with a fitted normal curve overlay
log_cap = np.log1p(df['capacity_mw'].values)
mu, sigma = np.mean(log_cap), np.std(log_cap)
x_range = np.linspace(log_cap.min(), log_cap.max(), 300)
normal_curve = (1 / (sigma * np.sqrt(2 * np.pi))) * np.exp(-0.5 * ((x_range - mu) / sigma) ** 2)

fig, ax = plt.subplots(figsize=(10, 5))
ax.hist(log_cap, bins=60, density=True, color='steelblue', edgecolor='white',
        alpha=0.7, label='log(1 + Capacity MW)')
ax.plot(x_range, normal_curve, color='crimson', linewidth=2, label='Normal fit')
ax.axvline(mu, color='orange', linestyle='--', linewidth=1.5, label=f'Mean = {mu:.2f}')
ax.set_title('Distribution of Power Plant Capacity (log scale)')
ax.set_xlabel('log(1 + Capacity MW)')
ax.set_ylabel('Density')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()